# Commodity-Linked Equity Portfolios
## CRSP / Compustat via WRDS - Fama-French Portfolio Construction

---

### What this notebook does

Constructs Block 2 commodity-linked equity portfolios for the test asset universe
(Table 1, N=78). Seven overlapping sort schemes are applied to the **same pooled
commodity-linked equity universe** (direct producers + processors, ~742 firm-years/yr),
following the MST (2025) approach of multiple characteristic sorts on one universe.

**Returns are weekly (Wednesday-to-Wednesday)**. Sort assignments are updated monthly
(HP-proxy, Basis-proxy, Momentum) or annually at June (BM). Daily CRSP returns are
compounded to Wednesday-to-Wednesday excess returns in Sections 9-11.

| Step | Description |
|------|-------------|
| **1** | Define commodity SIC code universe (producers + processors) |
| **2** | Connect to WRDS and pull CRSP + Compustat data |
| **3** | Compute book equity (Davis, Fama, French 2000 hierarchy) |
| **4** | Apply timing conventions - match BE to ME |
| **5** | Pull and clean CFTC COT data for HP-proxy |
| **6** | Compute equity momentum (12m-1m, vectorised); assign HP + Basis proxies |
| **7** | Build monthly sort panel (vectorised expansion) |
| **8** | Run 7 overlapping sort schemes (pooled universe) |
| **9** | Compute value-weighted daily returns |
| **10** | Aggregate daily --> **weekly Wednesday** returns |
| **11** | Subtract risk-free rate --> weekly excess returns |
| **12** | Diagnostics |
| **13** | Save outputs |
| **14** | Supplement: French 49 Industry Portfolios (Block 3) |

### Block 2 sort schemes (all on the same ~742 firm pooled universe)

| # | Sort | Grid | N | Factor alignment |
|---|------|------|---|------------------|
| 1 | BM x HP-proxy | 3x3 | 9 | Value x Hedging Pressure |
| 2 | MOM x BM | 3x3 | 9 | Momentum x Value |
| 3 | HP-proxy x MOM | 3x3 | 9 | Hedging Pressure x Momentum |
| 4 | BM x Basis-proxy | 3x3 | 9 | Value x Basis |
| 5 | BM quintiles | 1x5 | 5 | Value |
| 6 | MOM quintiles | 1x5 | 5 | Momentum |
| 7 | HP-proxy quintiles | 1x5 | 5 | Hedging Pressure |
| **Total** | | | **51** | |

### Outputs
- `commodity_equity_weekly.csv` - weekly excess returns, one column per portfolio (51 cols)
- `portfolio_counts_block2.csv` - stock counts per sort scheme per period
- `french_industry_weekly.csv` - Block 3 French 49-industry supplement (12 industries)


## 0 · Configuration

In [ ]:
import wrds
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import io
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

# User settings
WRDS_USER  = 'fdxavierj5'
START_DATE = '1989-07-01'   # one year before portfolio start (needs t-1 BE)
END_DATE   = '2023-12-31'
CHUNK_SIZE = 500             # permnos per WRDS query batch

# Minimum stocks per portfolio cell
MIN_STOCKS_DOUBLE = 5    # for 3x3 double sorts
MIN_STOCKS_SINGLE = 3    # for quintile single sorts

print("Configuration set.")
print(f"  Sample         : {START_DATE} --> {END_DATE}")
print(f"  Block 2 design : 7 overlapping sorts, pooled direct-producer universe")
print(f"  Block 2 target : 4x9 + 3x5 = 51 portfolios")

Configuration set.
  Sample         : 1989-07-01 → 2023-12-31
  Block 2 design : 7 overlapping sorts, pooled direct-producer universe
  Block 2 target : 4x9 + 3x5 = 51 portfolios


---
## 1 · Commodity SIC Code Universe

**Block 2 uses the full commodity-linked equity universe**: direct producers
(Oil_Gas, MetalMines, Coal, Agriculture, OtherMines) **plus** commodity processors
(Food, Textiles, Paper, Chemicals, RefinedPetroleum, Rubber, BldMaterials,
Steel_Metals, Utilities). All sorts apply to this pooled universe.

Processors are included here (unlike in the thesis design note) because the pooled
double/quintile sorts operate across a wide cross-section, and the SIC-to-commodity
mapping assigns each processor firm the HP and Basis of its primary input commodity,
giving them economically motivated exposure variation.


In [ ]:
# Block 2: full commodity-linked equity universe
# Includes both direct producers (revenue-side) and processors (cost-side).
# All 7 sort schemes are applied to this pooled universe.

""" SIC_MAP = {
    # Direct commodity producers
    "Agriculture": list(range(100, 1000)),  # Farming, fishing, forestry
    "MetalMines": list(range(1000, 1200)),  # Gold, silver, copper, iron ore
    "Coal": list(range(1200, 1300)),  # Bituminous + anthracite coal
    "Oil_Gas": list(range(1300, 1400)),  # Crude petroleum + natural gas Extraction
    "OtherMines": list(range(1400, 1500)),  # Stone, sand, potash, sulfur
    # Commodity processors
    "Food_Tobacco": list(range(2000, 2200)),  # Food, beverages, and tobacco processing
    "Textiles": list(range(2200, 2400)),  # Textiles, apparel, knitting mills
    "Lumber_Wood": list(range(2400, 2600)),  # Lumber, wood products, furniture
    "Paper": list(range(2600, 2700)),  # Paper and allied products
    "Chemicals": list(range(2800, 2900)),  # Chemicals and allied products
    "RefinedPetroleum": list(range(2900, 3000)),  # Petroleum refining and refining industries
    "Rubber_Plastics": list(range(3000, 3200)),  # Rubber and plastics fabrication
    "BldMaterials": list(range(3200, 3300)),  # Cement, glass, clay, ceramics
    "Steel_Metals": list(range(3300, 3400)),  # Blast furnaces, steel works, primary metals
    "FabMetals": list(range(3400, 3500)),  # Fabricated metal products (except machinery)
    "Utilities": list(range(4900, 4940)),  # Electric and Gas utilities
} 
"""

# SIC --> Sector mapping (non-overlapping, explicit ranges)
SIC_MAP = {
    # Energy
    'Oil_Gas':          [(1300, 1320), (1322, 1399)],   # crude producers & services
    'NatGasProd':       [(1321, 1321)],                 # natural gas liquids
    'Coal':             [(1200, 1299)],
    # Mining
    'GoldMines':        [(1040, 1049)],                 # gold & silver ores
    'CopperMines':      [(1060, 1069)],                 # ferroalloy ores
    'MetalMines':       [(1000, 1039), (1050, 1059),
                         (1070, 1199)],                 # other metal mining
    'OtherMines':       [(1400, 1499)],
    # Agriculture
    'CottonFarms':      [(131, 132)],
    'Livestock':        [(200, 259)],
    'Agriculture':      [(100, 130), (133, 199),        # grains + general farming
                         (260, 999)],
    # Food Processing
    'MeatProc':         [(2011, 2013)],
    'GrainMilling':     [(2041, 2048)],
    'SugarProc':        [(2060, 2068)],
    'OilsFats':         [(2075, 2079)],
    'Food':             [(2000, 2010), (2014, 2040),    # other food processing
                         (2049, 2059), (2069, 2074),
                         (2080, 2099)],
    'Tobacco':          [(2100, 2199)],
    # Other Processors & Utilities
    'Textiles':         [(2200, 2389)],
    'LumberWood':       [(2400, 2599)],
    'Paper':            [(2600, 2699)],
    'Chemicals':        [(2800, 2899)],
    'RefinedPetroleum': [(2900, 2999)],
    'Rubber':           [(3000, 3099)],
    'Plastics':         [(3100, 3199)],
    'BldMaterials':     [(3200, 3299)],
    'AluminumProc':     [(3353, 3356), (3460, 3469)],
    'Steel_Metals':     [(3300, 3352), (3357, 3459)],
    'FabMetals':        [(3400, 3499)],
    'Utilities':        [(4900, 4939)],
}

# Build flat SIC --> sector lookup
SIC_TO_SECTOR = {}
for sector, ranges in SIC_MAP.items():
    for start, end in ranges:
        for sic in range(start, end + 1):
            SIC_TO_SECTOR[sic] = sector

# Commodity assignment: 13 distinct commodities
SIC_TO_COMMODITY = {
    'Oil_Gas':          'WTI CRUDE OIL',
    'NatGasProd':       'NATURAL GAS',
    'Coal':             'HEATING OIL',
    'GoldMines':        'GOLD',
    'CopperMines':      'COPPER',
    'MetalMines':       'GOLD',
    'OtherMines':       'COPPER',
    'CottonFarms':      'COTTON NO. 2',
    'Livestock':        'LIVE CATTLE',
    'Agriculture':      'SOYBEANS',
    'MeatProc':         'LIVE CATTLE',
    'GrainMilling':     'WHEAT',
    'SugarProc':        'SUGAR NO. 11',
    'OilsFats':         'SOYBEANS',
    'Food':             'CORN',
    'Textiles':         'COTTON NO. 2',
    'Paper':            'LUMBER',
    'Chemicals':        'NATURAL GAS',
    'RefinedPetroleum': 'WTI CRUDE OIL',
    'Rubber':           'WTI CRUDE OIL',
    'BldMaterials':     'COPPER',
    'AluminumProc':     'COPPER', # aluminum has too few hedging pressure observations
    'Steel_Metals':     'COPPER',
    'Utilities':        'NATURAL GAS',
    'Tobacco':    'CORN',          # food/agri processing
    'LumberWood': 'LUMBER',        # direct lumber link
    'FabMetals':  'COPPER',        # fabricated metals --> copper
    'Plastics':   'WTI CRUDE OIL', # petrochemical inputs
}

distinct = sorted(set(SIC_TO_COMMODITY.values()))
print(f"Sectors: {len(SIC_MAP)}  |  Distinct commodities: {len(distinct)}")
for c in distinct:
    sectors = [s for s, cm in SIC_TO_COMMODITY.items() if cm == c]
    print(f"  {c:<22}: {', '.join(sectors)}")

Sectors: 28  |  Distinct commodities: 12
  COPPER                : CopperMines, OtherMines, BldMaterials, AluminumProc, Steel_Metals, FabMetals
  CORN                  : Food, Tobacco
  COTTON NO. 2          : CottonFarms, Textiles
  GOLD                  : GoldMines, MetalMines
  HEATING OIL           : Coal
  LIVE CATTLE           : Livestock, MeatProc
  LUMBER                : Paper, LumberWood
  NATURAL GAS           : NatGasProd, Chemicals, Utilities
  SOYBEANS              : Agriculture, OilsFats
  SUGAR NO. 11          : SugarProc
  WHEAT                 : GrainMilling
  WTI CRUDE OIL         : Oil_Gas, RefinedPetroleum, Rubber, Plastics


---
## 2 · Connect to WRDS and Pull Data

Same WRDS queries as before; restricted to direct-producer SIC codes only.


In [6]:
db = wrds.Connection(wrds_username=WRDS_USER)
print("Connected to WRDS.")

Loading library list...
Done
Connected to WRDS.


In [7]:
# 2A. Identify commodity permnos
ALL_COMMODITY_SIC = [
    sic for sic, sector in SIC_TO_SECTOR.items()
    if sector in SIC_TO_COMMODITY       
]

sic_str = ','.join(str(s) for s in ALL_COMMODITY_SIC)

names_query = f"""
    SELECT DISTINCT permno, siccd, shrcd, exchcd, namedt, nameendt
    FROM crsp.msenames
    WHERE siccd   IN ({sic_str})
      AND shrcd   IN (10, 11)
      AND exchcd  IN (1, 2, 3)
"""
names = db.raw_sql(names_query, date_cols=['namedt', 'nameendt'])
names['sector'] = names['siccd'].map(SIC_TO_SECTOR)

commodity_permnos = names['permno'].unique().tolist()
print(f"Found {len(commodity_permnos):,} unique direct-producer permnos")
print("Sector breakdown:")
print(names.drop_duplicates('permno')['sector'].value_counts().to_string())

Found 6,309 unique direct-producer permnos
Sector breakdown:
sector
Chemicals           1387
Oil_Gas             1073
FabMetals            507
Textiles             440
Food                 429
Utilities            354
Steel_Metals         336
Rubber               242
LumberWood           239
Paper                204
BldMaterials         170
RefinedPetroleum     143
MetalMines           138
GoldMines            114
Agriculture           84
Plastics              81
Coal                  66
SugarProc             55
MeatProc              54
OtherMines            44
Tobacco               41
GrainMilling          38
Livestock             21
AluminumProc          15
NatGasProd            13
OilsFats              10
CopperMines            8
CottonFarms            3


In [8]:
# 2B. CRSP monthly stock file
print("Pulling CRSP monthly stock file...")
monthly_chunks = []

for i in range(0, len(commodity_permnos), CHUNK_SIZE):
    chunk    = commodity_permnos[i:i + CHUNK_SIZE]
    perm_str = ','.join(str(p) for p in chunk)
    q = f"""
        SELECT a.permno, a.date, a.ret, a.retx, a.shrout, a.prc
        FROM crsp.msf AS a
        WHERE a.permno IN ({perm_str})
          AND a.date BETWEEN '{START_DATE}' AND '{END_DATE}'
    """
    monthly_chunks.append(db.raw_sql(q, date_cols=['date']))
    if (i // CHUNK_SIZE) % 5 == 0:
        pct = min(i + CHUNK_SIZE, len(commodity_permnos))
        print(f"  Monthly: {pct}/{len(commodity_permnos)} permnos pulled")

msf = pd.concat(monthly_chunks, ignore_index=True)
msf['me']    = msf['prc'].abs() * msf['shrout']
msf['year']  = msf['date'].dt.year
msf['month'] = msf['date'].dt.month

print(f"\nMonthly file: {len(msf):,} obs | "
      f"{msf['permno'].nunique():,} permnos | "
      f"{msf['date'].min().date()} - {msf['date'].max().date()}")

Pulling CRSP monthly stock file...
  Monthly: 500/6309 permnos pulled
  Monthly: 3000/6309 permnos pulled
  Monthly: 5500/6309 permnos pulled

Monthly file: 576,163 obs | 3,803 permnos | 1989-07-31 - 2023-12-29


In [9]:
# 2C. CRSP daily stock file
print("Pulling CRSP daily stock file (this takes several minutes, roughly 15-20min)...")
daily_chunks = []

for i in range(0, len(commodity_permnos), CHUNK_SIZE):
    chunk    = commodity_permnos[i:i + CHUNK_SIZE]
    perm_str = ','.join(str(p) for p in chunk)
    q = f"""
        SELECT permno, date, ret, prc, shrout, vol
        FROM crsp.dsf
        WHERE permno IN ({perm_str})
          AND date BETWEEN '1990-01-01' AND '{END_DATE}'
    """
    daily_chunks.append(db.raw_sql(q, date_cols=['date']))
    if (i // CHUNK_SIZE) % 5 == 0:
        pct = min(i + CHUNK_SIZE, len(commodity_permnos))
        print(f"  Daily: {pct}/{len(commodity_permnos)} permnos pulled")

dsf = pd.concat(daily_chunks, ignore_index=True)
dsf['year']  = dsf['date'].dt.year
dsf['month'] = dsf['date'].dt.month

print(f"\nDaily file: {len(dsf):,} obs | "
      f"{dsf['permno'].nunique():,} permnos | "
      f"{dsf['date'].min().date()} - {dsf['date'].max().date()}")

Pulling CRSP daily stock file (this takes several minutes, roughly 15-20min)...
  Daily: 500/6309 permnos pulled
  Daily: 3000/6309 permnos pulled
  Daily: 5500/6309 permnos pulled

Daily file: 11,786,323 obs | 3,733 permnos | 1990-01-02 - 2023-12-29


In [10]:
# 2D. Compustat annual fundamentals
print("Pulling Compustat annual fundamentals...")

comp_query = f"""
    SELECT gvkey, datadate, fyear,
           seq, ceq, pstk, at, lt,
           txditc, txdb,
           pstkrv, pstkl,
           sich
    FROM comp.funda
    WHERE datadate BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
      AND sich IN ({sic_str})
"""
comp = db.raw_sql(comp_query, date_cols=['datadate'])
comp = comp.rename(columns={'sich': 'sic'})

print(f"Compustat: {len(comp):,} obs | {comp['gvkey'].nunique():,} firms")

Pulling Compustat annual fundamentals...
Compustat: 81,106 obs | 8,182 firms


In [11]:
# 2E. CCM link table
print("Pulling CCM link table...")

ccm = db.raw_sql("""
    SELECT gvkey, lpermno AS permno,
           linkdt, linkenddt, linktype, linkprim
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC')
      AND linkprim IN ('P', 'C')
""", date_cols=['linkdt', 'linkenddt'])

ccm['linkenddt'] = ccm['linkenddt'].fillna(pd.Timestamp('2099-12-31'))
print(f"CCM: {len(ccm):,} links")

db.close()
print("WRDS connection closed.")

Pulling CCM link table...
CCM: 33,324 links
WRDS connection closed.


---
## 3 · Compute Book Equity

Follows the **Davis, Fama, French (2000)** hierarchy.

$$BE = \text{Stockholders' Equity} + \text{Deferred Taxes} - \text{Preferred Stock}$$

| Component | Hierarchy |
|-----------|-----------|
| Stockholders' Equity | `seq` --> `ceq + pstk` --> `at - lt` |
| Deferred Taxes | `txditc` --> `txdb` --> 0 |
| Preferred Stock | `pstkrv` --> `pstkl` --> `pstk` --> 0 |


In [12]:
def compute_book_equity(df):
    df = df.copy()

    # Stockholders' equity
    df['se'] = np.where(df['seq'].notna(), df['seq'],
                    np.where((df['ceq'].notna()) & (df['pstk'].notna()),
                             df['ceq'] + df['pstk'],
                    np.where((df['at'].notna()) & (df['lt'].notna()),
                             df['at'] - df['lt'], np.nan)))

    # Deferred taxes
    df['dt'] = np.where(df['txditc'].notna(), df['txditc'],
                    np.where(df['txdb'].notna(), df['txdb'], 0))

    # Preferred stock
    df['ps'] = np.where(df['pstkrv'].notna(), df['pstkrv'],
                    np.where(df['pstkl'].notna(), df['pstkl'],
                        np.where(df['pstk'].notna(), df['pstk'], 0)))

    df['be'] = df['se'] + df['dt'] - df['ps']
    df.loc[df['be'] <= 0, 'be'] = np.nan

    return df[['gvkey', 'datadate', 'fyear', 'be', 'sic']]

comp_be = compute_book_equity(comp)

valid = comp_be['be'].notna().sum()
total = len(comp_be)
print(f"Book equity: {valid:,} valid obs ({valid/total:.1%}) out of {total:,}")

Book equity: 70,431 valid obs (86.8%) out of 81,106


---
## 4 · Timing Conventions - Match BE to ME

Standard Fama-French timing:

| Variable | Measurement date | Purpose |
|----------|-----------------|---------|
| **BE** | Fiscal year ending in calendar year **t−1** | BM numerator |
| **ME (December)** | December **t−1** | BM denominator |
| **ME (June)** | June **t** | Size weight for VW returns |

Portfolios formed at **end of June year t**, held **July t - June t+1**.


In [13]:
# 4A. Assign portfolio_year to Compustat
comp_be['fiscal_year_end_year'] = comp_be['datadate'].dt.year
comp_be['portfolio_year']       = comp_be['fiscal_year_end_year'] + 1

comp_be = (comp_be
           .sort_values(['gvkey', 'portfolio_year', 'datadate'])
           .groupby(['gvkey', 'portfolio_year'])
           .last()
           .reset_index())

print(f"Compustat after dedup: {len(comp_be):,} firm-portfolio_year obs")

Compustat after dedup: 80,932 firm-portfolio_year obs


In [ ]:
# 4B. Link Compustat --> CRSP via CCM
def link_comp_to_crsp(comp_df, ccm_df, date_col='datadate'):
    merged = comp_df.merge(ccm_df, on='gvkey', how='left')
    valid  = ((merged[date_col] >= merged['linkdt']) &
              (merged[date_col] <= merged['linkenddt']))
    merged = merged[valid].copy()
    merged = (merged
              .sort_values('linkprim', ascending=False)
              .groupby(['gvkey', date_col])
              .first()
              .reset_index())
    return merged

comp_linked = link_comp_to_crsp(comp_be, ccm, date_col='datadate')
comp_linked = comp_linked[['permno', 'portfolio_year', 'be']].dropna()

print(f"CCM-linked: {len(comp_linked):,} obs | "
      f"{comp_linked['permno'].nunique():,} permnos")

CCM-linked: 50,244 obs | 5,408 permnos


In [ ]:
# 4C. December ME for BM denominator
dec_me = (msf[msf['month'] == 12]
          [['permno', 'year', 'me']]
          .rename(columns={'me': 'me_dec', 'year': 'dec_year'})
          .assign(portfolio_year=lambda d: d['dec_year'] + 1))

# 4D. June ME for VW weights
jun_me = (msf[msf['month'] == 6]
          [['permno', 'year', 'me', 'date']]
          .rename(columns={'me': 'me_jun', 'year': 'portfolio_year'}))

# 4E. Merge sort variables
sort_data = (jun_me
             .merge(dec_me[['permno', 'portfolio_year', 'me_dec']],
                    on=['permno', 'portfolio_year'], how='left')
             .merge(comp_linked,
                    on=['permno', 'portfolio_year'], how='left')
             .merge(names[['permno', 'siccd', 'namedt', 'nameendt', 'sector']]
                    .drop_duplicates(),
                    on='permno', how='left'))

# Validate SIC code at June date
sort_data = sort_data[
    (sort_data['date'] >= sort_data['namedt']) &
    (sort_data['date'] <= sort_data['nameendt'])
]
sort_data['sector'] = sort_data['siccd'].map(SIC_TO_SECTOR)

# BM ratio (Compustat BE is millions; CRSP ME is thousands --> convert ME to millions)
sort_data['me_dec_m'] = sort_data['me_dec'] / 1000  # thousands --> millions
sort_data['bm'] = sort_data['be'] / sort_data['me_dec_m']

# Drop missing/invalid
sort_data = sort_data.dropna(subset=['me_jun', 'bm'])
sort_data = sort_data[(sort_data['me_jun'] > 0) & (sort_data['bm'] > 0)]

print(f"Sort data ready: {len(sort_data):,} firm-year obs")
print(f"Portfolio years: {sort_data['portfolio_year'].min()} - "
      f"{sort_data['portfolio_year'].max()}")
print(f"Avg stocks/year: {sort_data.groupby('portfolio_year').size().mean():.0f}")

Sort data ready: 29,383 firm-year obs
Portfolio years: 1990 - 2023
Avg stocks/year: 864


---
## 5 · CFTC COT Data - Hedging Pressure and Basis Proxies

Download CFTC Legacy Combined COT reports (annual CSV files, 1995-2023).  
Extract the net commercial short position for each mapped commodity.

HP = (Short_comm − Long_comm) / (Short_comm + Long_comm)  
Positive HP --> commercial producers are net short (high hedging demand).

Basis-proxy: sourced from commodity futures prices (Bloomberg/CFTC).  
Here we construct a placeholder series; replace with futures data when available.


In [ ]:
# 5A. Load COT data: two-source splice
#
# Legacy Futures-Only    (1995-May 2009): Commercial = PMPU + Swap Dealers
# Disaggregated Fut-Only (Jun 2009-2023): PMPU only (pure physical hedgers)
#
# Manual download (if API fails - two separate files needed):
#   Legacy FutOnly : https://publicreporting.cftc.gov/Commitments-of-Traders/Legacy-Futures-Only/rh2e-ybqs
#                   --> save as cot_legacy_futonly.csv
#   Disagg FutOnly : https://publicreporting.cftc.gov/Commitments-of-Traders/Disaggregated-Futures-Only/72hh-3qpy
#                   --> save as cot_disagg_futonly.csv
#
# Then set the local_csv fields below and re-run.

import requests, time

SPLICE_DATE = '2009-06-01'

SOURCES = {
    'legacy': dict(
        endpoint   = 'https://publicreporting.cftc.gov/resource/rh2e-ybqs.json',
        local_csv  = 'cot_legacy_futonly.csv',           # ← 'cot_legacy_futonly.csv' if API fails
        date_col   = 'report_date_as_yyyy_mm_dd',
        name_col   = 'contract_market_name',
        long_col   = 'comm_positions_long_all',
        short_col  = 'comm_positions_short_all',
        date_start = '1995-01-01',
        date_end   = '2009-05-31',
        label      = 'Legacy FutOnly',
    ),
    'disagg': dict(
        endpoint   = 'https://publicreporting.cftc.gov/resource/72hh-3qpy.json',
        local_csv  = 'cot_disagg_futonly.csv',           # ← 'cot_disagg_futonly.csv' if API fails
        date_col   = 'report_date_as_yyyy_mm_dd',
        name_col   = 'contract_market_name',
        long_col   = 'prod_merc_positions_long_all',
        short_col  = 'prod_merc_positions_short_all',
        date_start = '2009-06-01',
        date_end   = '2023-12-31',
        label      = 'Disaggregated FutOnly (PMPU)',
    ),
}

# Broad keywords used for contract filtering in both sources
FILTER_KEYWORDS = [
    'CRUDE', 'WTI',
    'NATURAL GAS', 'NAT GAS NYME',
    'HEATING OIL', 'ULSD', 'HO', 'HGO', 'NO. 2 FUEL OIL',
    'GOLD', 'SILVER', 'COPPER',
    'SOYBEAN', 'CORN', 'WHEAT',
    'COTTON', 'SUGAR', 'COFFEE', 'COCOA',
    'CATTLE', 'HOGS',
    'LUMBER',
]


def _load_cot_source(cfg):
    """
    Load one COT source (Legacy or Disaggregated) via Socrata API or local CSV.
    Returns DataFrame with standardised columns:
        date, name, long, short
    covering cfg['date_start'] to cfg['date_end'].
    """
    start, end = cfg['date_start'], cfg['date_end']

    # Local CSV path
    if cfg['local_csv']:
        print(f"  {cfg['label']}: loading from {cfg['local_csv']}...")

        # Peek at header to build column map
        header = pd.read_csv(cfg['local_csv'], nrows=0)
        lc_map = {c.lower().replace(' ', '_'): c for c in header.columns}

        def find_col(key):
            # try exact key first, then case-insensitive
            return lc_map.get(key, lc_map.get(key.lower(), None))

        actual = {
            'date' : find_col(cfg['date_col']),
            'name' : find_col(cfg['name_col']),
            'long' : find_col(cfg['long_col']),
            'short': find_col(cfg['short_col']),
        }
        missing = {k: v for k, v in actual.items() if v is None}
        if missing:
            raise ValueError(
                f"{cfg['label']}: could not find columns {list(missing.keys())}.\n"
                f"Available: {header.columns.tolist()}"
            )

        chunks = []
        for chunk in pd.read_csv(cfg['local_csv'],
                                  usecols=list(actual.values()),
                                  chunksize=50_000, low_memory=False,
                                  thousands=','):
            chunk = chunk.rename(columns={v: k for k, v in actual.items()})
            chunk['date'] = pd.to_datetime(
                chunk['date'], infer_datetime_format=True, errors='coerce')
            chunk = chunk[chunk['date'].between(start, end)]
            upper = chunk['name'].str.upper()
            keep  = sum(upper.str.contains(kw, na=False)
                        for kw in FILTER_KEYWORDS).gt(0)
            if keep.any():
                chunks.append(chunk[keep])

        df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(
            columns=['date', 'name', 'long', 'short'])

    # Socrata API path
    else:
        print(f"  {cfg['label']}: querying Socrata API...")
        all_chunks = []

        # One request per keyword - avoids IN() and LIKE issues with
        # special characters in Socrata SoQL
        for kw in FILTER_KEYWORDS:
            where  = (f"{cfg['name_col']} LIKE '%{kw}%'"
                      f" AND {cfg['date_col']} >= '{start}'"
                      f" AND {cfg['date_col']} <= '{end}'")
            select = ','.join([cfg['date_col'], cfg['name_col'],
                               cfg['long_col'], cfg['short_col']])
            offset = 0
            while True:
                r = requests.get(
                    cfg['endpoint'],
                    params={'$where': where, '$select': select,
                            '$limit': 50000, '$offset': offset},
                    timeout=60
                )
                r.raise_for_status()
                batch = r.json()
                if not batch:
                    break
                all_chunks.append(pd.DataFrame(batch))
                if len(batch) < 50000:
                    break
                offset += 50000
                time.sleep(0.25)
            print(f"    '{kw}': {sum(len(c) for c in all_chunks):,} rows so far")

        if not all_chunks:
            raise RuntimeError(
                f"{cfg['label']}: Socrata returned no data.\n"
                f"Download manually - see instructions at top of cell."
            )

        df = pd.concat(all_chunks, ignore_index=True)
        df = df.rename(columns={
            cfg['date_col'] : 'date',
            cfg['name_col'] : 'name',
            cfg['long_col'] : 'long',
            cfg['short_col']: 'short',
        })
        df['date'] = pd.to_datetime(
            df['date'], infer_datetime_format=True, errors='coerce')
        df = df.drop_duplicates()   # keyword overlap can create dupes

    # Normalise numeric columns
    for col in ['long', 'short']:
        df[col] = (df[col].astype(str)
                          .str.replace(',', '', regex=False)
                          .str.strip()
                          .pipe(pd.to_numeric, errors='coerce'))

    df = df.dropna(subset=['date']).reset_index(drop=True)

    contracts = sorted(df['name'].dropna().unique().tolist())
    print(f"    Loaded: {len(df):,} rows | "
          f"{df['date'].min().date()} - {df['date'].max().date()}")
    print(f"    Contracts ({len(contracts)}): {contracts}")
    return df


# Load both
print("=" * 60)
cot_legacy = _load_cot_source(SOURCES['legacy'])
print("=" * 60)
cot_disagg  = _load_cot_source(SOURCES['disagg'])
print("=" * 60)
print("Both COT sources loaded.")

  Legacy FutOnly: loading from cot_legacy_futonly.csv...
    Loaded: 17,775 rows | 1995-01-03 - 2009-05-26
    Contracts (51): ['1000 TROY OUNCE SILVER', 'ALBERTA NATURAL GAS', 'COCOA', 'COFFEE C', 'COLUMBIA GULF ONSHORE', 'COPPER- #1', 'CORN', 'CORN in 1000 Bushels', 'COTTON NO. 2', 'CRUDE OIL, BRENT', 'DIAMMONIUM PHOSPHATE', 'DUBAI CRUDE OIL CALENDAR', 'E-MINI CRUDE OIL, LIGHT SWEET', 'E-MINI NATURAL GAS', 'FEEDER CATTLE', 'GOLD', 'GOLD, 100 TROY OZ', 'HARD AMBER DURUM WHEAT', 'HOUSTON SHIP CH BASIS', 'HOUSTON SHIP CH INDEX', 'IOWA CORN YIELD INSURANCE', 'LEAN HOGS', 'LIVE CATTLE', 'LIVE HOGS', 'MINI SOYBEANS', 'NAT GAS NYME', 'NATURAL GAS', 'NY HARBOR ULSD', 'NYMEX HEATING OIL/RDAM GASOIL', 'RANDOM LENGTH LUMBER', 'RANDOM LENGTH LUMBER-old', 'SILVER', 'SILVER, 5000 TROY OZ', 'SOYBEAN MEAL', 'SOYBEAN OIL', 'SOYBEANS', 'SOYBEANS in 1000 Bushels', 'SUGAR NO. 11', 'SUGAR NO. 14', 'UP DOWN GC ULSD VS HO SPR', 'US CORN YIELD INSURANCE', 'WHEAT', 'WHEAT in 1000 Bushels', 'WHEAT-HRSpring', 

In [ ]:
# 5B. Compute spliced, z-scored HP series
#
# For each commodity:
#   1. Aggregate longs + shorts across all related contracts per date
#      (captures full hedging demand regardless of which contract is used)
#   2. HP = (short - long) / (short + long)
#   3. Apply rolling 52-week z-score WITHIN each sub-period separately
#      --> removes the level shift from stripping swap dealers at the splice
#   4. Concatenate Legacy (pre-Jun 2009) + Disaggregated (Jun 2009+)
#
# Note on Legacy contract names: Legacy FutOnly may use older contract names
# (e.g. 'CRUDE OIL, LIGHT SWEET' without '-WTI'). These are included below
# and will simply return 0 rows if not present in the file - no harm done.
COT_CONTRACT_MAP = {
    'WTI CRUDE OIL': {
        'legacy': ['WTI-PHYSICAL', 'WTI CRUDE OIL FINANCIAL', 
                   'WTI FINANCIAL CRUDE OIL', 'E-MINI CRUDE OIL, LIGHT SWEET'],
        'disagg': ['CRUDE OIL, LIGHT SWEET-WTI', 'WTI CRUDE OIL 1ST LINE',
                   'WTI FINANCIAL CRUDE OIL', 'WTI CRUDE OIL FINANCIAL'],
    },
    'HEATING OIL': {
        'legacy': ['NYMEX HEATING OIL/RDAM GASOIL', 'NY HARBOR ULSD'],
        'disagg': ['NYMEX HEATING OIL/RDAM GASOIL', 'NY HARBOR ULSD', 'GULF COAST ULSD-PLATTS'],
    },
    'NATURAL GAS': {
        'legacy': ['NATURAL GAS', 'E-MINI NATURAL GAS', 'NAT GAS NYME'],
        'disagg': ['NAT GAS NYME'],   # no Henry Hub contract in Disagg file; ffill from Legacy
    },
    'GOLD': {
        'legacy': ['GOLD', 'GOLD, 100 TROY OZ'],
        'disagg': ['GOLD', 'MICRO GOLD'],
    },
    'COPPER': {
        'legacy': ['COPPER- #1'],
        'disagg': ['COPPER- #1'],
    },
    'SOYBEANS': {
        'legacy': ['SOYBEANS', 'SOYBEANS in 1000 Bushels'],
        'disagg': ['SOYBEANS'],
    },
    'CORN': {
        'legacy': ['CORN', 'CORN in 1000 Bushels'],
        'disagg': ['CORN', 'MINI CORN'],
    },
    'COTTON NO. 2': {
        'legacy': ['COTTON NO. 2'],
        'disagg': ['COTTON NO. 2'],
    },
    'LUMBER': {
        'legacy': ['RANDOM LENGTH LUMBER', 'RANDOM LENGTH LUMBER-old'],
        'disagg': ['LUMBER', 'RANDOM LENGTH LUMBER'],
    },
    'LIVE CATTLE': {
        'legacy': ['LIVE CATTLE'],
        'disagg': ['LIVE CATTLE'],
    },
    'WHEAT': {
        'legacy': ['WHEAT', 'WHEAT in 1000 Bushels', 'WHEAT-HRW', 'WHEAT-SRW'],
        'disagg': ['WHEAT-HRW', 'WHEAT-SRW', 'WHEAT-HRSpring'],
    },
    'SUGAR NO. 11': {
        'legacy': ['SUGAR NO. 11'],
        'disagg': ['SUGAR NO. 11'],
    }
}


def aggregate_hp(cot_df, contract_names):
    """
    Sum commercial longs and shorts across all listed contracts per report date,
    then compute HP = (short - long) / (short + long).
    Returns a pd.Series indexed by date.
    """
    present = [n for n in contract_names if n in cot_df['name'].values]
    if not present:
        return pd.Series(dtype=float)

    sub = cot_df[cot_df['name'].isin(present)].copy()
    agg = (sub.groupby('date')[['long', 'short']]
              .sum(min_count=1)   # NaN if ALL contracts NaN for that date
              .reset_index())

    denom = (agg['short'] + agg['long']).replace(0, np.nan)
    hp    = (agg['short'] - agg['long']) / denom

    return pd.Series(hp.values,
                     index=pd.to_datetime(agg['date'])).dropna().sort_index()


def rolling_zscore(series, window=52, min_periods=26):
    """
    Rolling z-score: (x - rolling_mean) / rolling_std.
    min_periods=26 --> first valid z-score after ~6 months.
    Applied within each sub-period so the splice does not introduce jumps.
    """
    m = series.rolling(window, min_periods=min_periods).mean()
    s = series.rolling(window, min_periods=min_periods).std()
    return ((series - m) / s.replace(0, np.nan)).dropna()


hp_records = []

print("Computing spliced HP (rolling 52w z-score)...\n")

for commodity_key, names in COT_CONTRACT_MAP.items():

    # Raw HP per sub-period
    hp_leg  = aggregate_hp(cot_legacy, names['legacy'])
    hp_disg = aggregate_hp(cot_disagg,  names['disagg'])

    if hp_leg.empty and hp_disg.empty:
        print(f"  WARNING: no contracts matched for {commodity_key}")
        print(f"    Names tried: {names}")
        print(f"    Legacy contracts available  : "
              f"{sorted(cot_legacy['name'].unique())[:10]}")
        print(f"    Disaggregated contracts avail: "
              f"{sorted(cot_disagg['name'].unique())[:10]}")
        continue

    # Rolling z-score within each sub-period
    z_leg  = rolling_zscore(hp_leg)  if not hp_leg.empty  else pd.Series(dtype=float)
    z_disg = rolling_zscore(hp_disg) if not hp_disg.empty else pd.Series(dtype=float)

    # Splice
    combined = (pd.concat([z_leg, z_disg])
                  .sort_index()
                  .pipe(lambda s: s[~s.index.duplicated(keep='last')]))

    hp_df = pd.DataFrame({
        'report_date': combined.index,
        'hp':          combined.values,
        'commodity':   commodity_key,
    }).dropna(subset=['hp'])

    hp_records.append(hp_df)

    # Coverage report
    print(f"  {commodity_key}")
    if not z_leg.empty:
        print(f"    Legacy  : {len(z_leg):>5} obs  "
              f"{z_leg.index.min().date()} - {z_leg.index.max().date()}  "
              f"mean={z_leg.mean():.2f}  std={z_leg.std():.2f}")
    else:
        print(f"    Legacy  : 0 obs - no matching contracts found")
    if not z_disg.empty:
        print(f"    Disagg  : {len(z_disg):>5} obs  "
              f"{z_disg.index.min().date()} - {z_disg.index.max().date()}  "
              f"mean={z_disg.mean():.2f}  std={z_disg.std():.2f}")
    else:
        print(f"    Disagg  : 0 obs - no matching contracts found")
    print(f"    Spliced : {len(hp_df):>5} obs  "
          f"{hp_df['report_date'].min().date()} - "
          f"{hp_df['report_date'].max().date()}")

hp_all = pd.concat(hp_records, ignore_index=True)
print(f"\nTotal spliced HP observations: {len(hp_all):,}")
print("Units: rolling 52-week z-score within each sub-period (mean≈0, std≈1).")

Computing spliced HP (rolling 52w z-score)...

  WTI CRUDE OIL
    Legacy  :   726 obs  1995-06-27 - 2009-05-26  mean=-0.18  std=1.13
    Disagg  :   736 obs  2009-11-24 - 2023-12-26  mean=0.01  std=1.28
    Spliced :  1462 obs  1995-06-27 - 2023-12-26
  HEATING OIL
    Legacy  :   726 obs  1995-06-27 - 2009-05-26  mean=-0.06  std=1.12
    Disagg  :   736 obs  2009-11-24 - 2023-12-26  mean=0.05  std=1.23
    Spliced :  1462 obs  1995-06-27 - 2023-12-26
  NATURAL GAS
    Legacy  :   727 obs  1995-06-27 - 2009-05-26  mean=-0.26  std=1.19
    Disagg  :   736 obs  2009-11-24 - 2023-12-26  mean=0.07  std=1.27
    Spliced :  1463 obs  1995-06-27 - 2023-12-26
  GOLD
    Legacy  :   727 obs  1995-06-27 - 2009-05-26  mean=0.00  std=1.14
    Disagg  :   736 obs  2009-11-24 - 2023-12-26  mean=-0.13  std=1.31
    Spliced :  1463 obs  1995-06-27 - 2023-12-26
  COPPER
    Legacy  :   727 obs  1995-06-27 - 2009-05-26  mean=-0.15  std=1.24
    Disagg  :   736 obs  2009-11-24 - 2023-12-26  mean=-0.19  

---
## 6 · Assign Firm-Level HP-Proxy and Basis-Proxy

Each firm is assigned the HP and Basis of its primary commodity via the
SIC-to-commodity mapping defined in Section 1.

- **HP-proxy**: the most recent available COT HP report as of portfolio formation month
- **Basis-proxy**: (F2 − F1) / F1 for the firm's primary commodity  
  (computed from LSEG Datastream near/next futures prices, see Section 6B)

Both variables are re-assigned **monthly** (not just annually at June).


In [18]:
# 6A. Build monthly HP lookup per unique commodity
# For each commodity in COT_CONTRACT_MAP, build a month-end HP series
# by forward-filling the weekly COT reports to calendar month-ends.

monthly_dates = pd.date_range('1995-01-31', '2023-12-31', freq='ME')

hp_monthly_by_commodity = {}

for commodity_key in COT_CONTRACT_MAP.keys():
    hp_comm = hp_all[hp_all['commodity'] == commodity_key].copy()
    if len(hp_comm) == 0:
        print(f"  WARNING: no HP data for {commodity_key}")
        hp_monthly_by_commodity[commodity_key] = pd.Series(
            np.nan, index=monthly_dates, name=commodity_key
        )
        continue
    hp_comm = hp_comm.set_index('report_date').sort_index()
    hp_monthly = hp_comm['hp'].reindex(monthly_dates, method='ffill')
    hp_monthly.name = commodity_key
    hp_monthly_by_commodity[commodity_key] = hp_monthly
    print(f"  {commodity_key}: {hp_monthly.notna().sum()} non-missing months")

# Note expect 5-month z-score burn-in period at start of sample (Jan to May 1995) 
# (splice gap (June-November 2009) is correctly handled by ffill --> those month-ends get forward-filled from the last valid Legacy value (May 2009) until the first Disaggregated z-score appears in November 2009. No data loss there.)
hp_monthly_df = pd.DataFrame(hp_monthly_by_commodity)
hp_monthly_df.index.name = 'month_end'
hp_monthly_df = hp_monthly_df.reset_index()
hp_monthly_df['year']  = hp_monthly_df['month_end'].dt.year
hp_monthly_df['month'] = hp_monthly_df['month_end'].dt.month

print(f"\nHP monthly lookup: {len(hp_monthly_df)} month-rows x "
      f"{len(COT_CONTRACT_MAP)} commodities")

  WTI CRUDE OIL: 343 non-missing months
  HEATING OIL: 343 non-missing months
  NATURAL GAS: 343 non-missing months
  GOLD: 343 non-missing months
  COPPER: 343 non-missing months
  SOYBEANS: 343 non-missing months
  CORN: 343 non-missing months
  COTTON NO. 2: 343 non-missing months
  LUMBER: 343 non-missing months
  LIVE CATTLE: 343 non-missing months
  WHEAT: 343 non-missing months
  SUGAR NO. 11: 343 non-missing months

HP monthly lookup: 348 month-rows x 12 commodities


In [ ]:
# 6B. Basis-proxy from LSEG Datastream futures prices
import pandas as pd
import numpy as np

FUTURES_FILE = 'commodity_futures_data.xlsx'

COMMODITY_FUTURES_MAP = {
    'WTI CRUDE OIL': ('NCLC.01', 'NCLC.02'),
    'HEATING OIL':   ('NHOC.01', 'NHOC.02'),
    'NATURAL GAS':   ('NNGC.01', 'NNGC.02'),
    'GOLD':          ('NGCC.01', 'NGCC.02'),
    'COPPER':        ('NHGC.01', 'NHGC.02'),
    'SOYBEANS':      ('CSYC.01', 'CSYC.02'),
    'CORN':          ('CCFC.01', 'CCFC.02'),
    'COTTON NO. 2':  ('NCTC.01', 'NCTC.02'),
    'LUMBER':        ('CLBC.01', 'CLBC.02'),
    'LIVE CATTLE':   ('CLDC.01', 'CLDC.02'),
    'WHEAT':         ('CWFC.01', 'CWFC.02'),
    'SUGAR NO. 11':  ('NSBC.01', 'NSBC.02'),
    'ALUMINUM':      ('LAHC.01', 'LAHC.02'),
}

# 1. Metadata
meta = pd.read_excel(FUTURES_FILE, header=None, nrows=6, dtype=str)
code_row = meta.iloc[4, :]
ps_positions, ps_codes = [], []
for pos, val in code_row.items():
    s = str(val) if pd.notna(val) else ''
    if '(PS)' in s:
        ps_positions.append(pos)
        ps_codes.append(s.replace('(PS)', '').strip())
print(f"PS series found ({len(ps_codes)}): {ps_codes[:6]} ...")

# 2. Load data
data = pd.read_excel(FUTURES_FILE, header=None, skiprows=6,
                     usecols=[0] + ps_positions, index_col=0)
data.index      = pd.to_datetime(data.index, errors='coerce')
data.index.name = 'date'
data.columns    = ps_codes
data = data.sort_index().apply(pd.to_numeric, errors='coerce')
print(f"Loaded: {data.shape[0]} daily obs × {data.shape[1]} series")
print(f"Period: {data.index.min().date()} – {data.index.max().date()}")

# 3. Fix data errors
for col in ['NHOC.01', 'NHOC.02']:
    if col in data.columns:
        data.loc['2009-10-06':'2009-10-07', col] = np.nan
        print(f"Fixed: {col} 2009-10-06/07 --> NaN")

for col in data.columns:
    s = data[col].dropna()
    ratio_prev = s / s.shift(1)
    ratio_next = s / s.shift(-1)
    spike = ((ratio_prev > 5) & (ratio_next > 5)) | \
            ((ratio_prev < 0.2) & (ratio_next < 0.2))
    if spike.any():
        print(f"  {col}: {spike.sum()} spike(s) --> NaN")
        data.loc[spike[spike].index, col] = np.nan

# 4. Compute monthly basis
basis_records = []
for commodity, (f1_code, f2_code) in COMMODITY_FUTURES_MAP.items():
    if f1_code not in data.columns or f2_code not in data.columns:
        print(f"  WARN  {commodity:<22} - {f1_code} or {f2_code} not found")
        continue
    f1    = data[f1_code].resample('ME').last()
    f2    = data[f2_code].resample('ME').last()
    basis = ((f2 - f1) / f1).rename(commodity)
    basis_records.append(basis)
    n = basis.notna().sum()
    print(f"  OK    {commodity:<22}: {n} months  "
          f"[{basis.min():.3f}, {basis.max():.3f}]")

# 5. Assemble
if basis_records:
    basis_all        = pd.concat(basis_records, axis=1).loc['1994-01':'2023-12']
    basis_monthly_df = (basis_all.reset_index()
                                  .rename(columns={'date': 'month_end'}))
    basis_monthly_df['year']  = basis_monthly_df['month_end'].dt.year
    basis_monthly_df['month'] = basis_monthly_df['month_end'].dt.month
    print(f"\nBasis proxy: {len(basis_monthly_df)} months × {len(basis_records)} commodities")
    print(f"Period: {basis_monthly_df['month_end'].min().date()} – "
          f"{basis_monthly_df['month_end'].max().date()}")
    print("\nMissing rates:")
    for col in basis_all.columns:
        print(f"  {col:<22}: {basis_all[col].isna().mean()*100:.1f}%")
    print("\nSample statistics (monthly basis):")
    print(basis_all.describe().round(4).to_string())
else:
    print("\nNo basis computed - check WARN messages above")
    basis_monthly_df = hp_monthly_df[['year', 'month', 'month_end']].copy()
    for comm in COT_CONTRACT_MAP.keys():
        basis_monthly_df[comm] = np.nan

PS series found (61): ['NCLC.01', 'NCLC.02', 'NNGC.01', 'NNGC.02', 'NHOC.01', 'NHOC.02'] ...
Loaded: 10436 daily obs × 61 series
Period: 1985-01-01 – 2024-12-31
Fixed: NHOC.01 2009-10-06/07 → NaN
Fixed: NHOC.02 2009-10-06/07 → NaN
  NCLC.01: 1 spike(s) → NaN
  LEDC.02: 2 spike(s) → NaN
  LTIC.02: 1 spike(s) → NaN
  OK    WTI CRUDE OIL         : 480 months  [-0.061, 0.197]
  OK    HEATING OIL           : 480 months  [-0.261, 0.138]
  OK    NATURAL GAS           : 417 months  [-0.251, 0.360]
  OK    GOLD                  : 480 months  [-0.002, 0.008]
  OK    COPPER                : 438 months  [-0.110, 0.020]
  OK    SOYBEANS              : 480 months  [-0.124, 0.029]
  OK    CORN                  : 480 months  [-0.230, 0.076]
  OK    COTTON NO. 2          : 480 months  [-0.537, 0.190]
  OK    LUMBER                : 459 months  [-0.195, 0.146]
  OK    LIVE CATTLE           : 480 months  [-0.143, 0.080]
  OK    WHEAT                 : 480 months  [-0.167, 0.075]
  OK    SUGAR NO. 11     

In [ ]:
# 6C. Equity momentum: 12m−1m (vectorised)
# Standard definition: cumulative return months t-12 to t-2 (skip t-1).
# shift(1) skips the most recent month; rolling(12) gives the prior 12 months.

msf_sorted = (msf[['permno', 'date', 'ret', 'year', 'month']]
              .dropna(subset=['ret'])
              .sort_values(['permno', 'date'])
              .copy())

# Vectorised 12m-1m momentum - no groupby-apply loop
log1r = np.log1p(msf_sorted['ret'])   # log(1+r) for additive compounding
msf_sorted['log1r'] = log1r

mom_vals = (
    msf_sorted
    .groupby('permno')['log1r']
    .transform(
        lambda x: x.shift(1)            # skip most recent month (t-1)
                   .rolling(12, min_periods=10)  # sum log-returns t-12 … t-2
                   .sum()
    )
)
msf_sorted['mom'] = np.expm1(mom_vals)   # convert back: exp(sum log) - 1

mom_data = msf_sorted[['permno', 'year', 'month', 'mom']].dropna(subset=['mom'])
print(f"Momentum data: {len(mom_data):,} firm-month obs")
print(f"Date range   : {msf_sorted['date'].min().date()} - "
      f"{msf_sorted['date'].max().date()}")

Momentum data: 518,021 firm-month obs
Date range   : 1989-07-31 - 2023-12-29


---
## 7 · Build Monthly Sort Panel

For each calendar month, assemble the full cross-section of all firms in the
direct-producer universe with valid BM, HP-proxy, Basis-proxy, and Momentum.

- **BM**: assigned at June formation, constant over July t - June t+1  
- **HP-proxy, Basis-proxy**: updated monthly from COT / futures data  
- **Momentum**: updated monthly from CRSP monthly returns  

All 7 sort schemes share this same monthly panel.


In [ ]:
# 7A. Build monthly sort panel (vectorised)
# For each firm-year formation obs (from sort_annual = sort_data),
# generate 12 monthly rows covering the holding period July t - June t+1.
# BM and me_jun are constant within the holding year; momentum, HP, Basis
# are updated each month.

# --- vectorised expansion ---
# Add year/month for all 12 holding months without iterrows
sort_annual = sort_data.copy()

# Build a lookup: portfolio_year t --> list of (year, month) for Jul t - Jun t+1
holding_months = pd.DataFrame(
    [(yr, y, m)
     for yr in sort_annual['portfolio_year'].unique()
     for y, m in
         [(yr, mo) for mo in range(7, 13)] +
         [(yr + 1, mo) for mo in range(1, 7)]],
    columns=['portfolio_year', 'year', 'month']
)

# Cross-join: each firm-year obs gets 12 month rows
monthly_panel = sort_annual[
    ['permno', 'portfolio_year', 'bm', 'me_jun', 'sector']
].merge(holding_months, on='portfolio_year', how='left')

print(f"Monthly panel (pre-merge): {len(monthly_panel):,} firm-month obs")

# 7B. Merge momentum
monthly_panel = monthly_panel.merge(
    mom_data[['permno', 'year', 'month', 'mom']],
    on=['permno', 'year', 'month'],
    how='left'
)

# 7C. Merge HP-proxy via commodity lookup
# Map each sector to its primary commodity key, then look up monthly HP.

# Melt hp_monthly_df to long format: (year, month, commodity, hp)
hp_long = hp_monthly_df.melt(
    id_vars=['year', 'month', 'month_end'],
    var_name='commodity',
    value_name='hp'
)

hp_long['year']  = hp_long['year'].astype('int64')
hp_long['month'] = hp_long['month'].astype('int64')

# Add commodity column to monthly_panel via SIC_TO_COMMODITY
monthly_panel['commodity'] = monthly_panel['sector'].map(SIC_TO_COMMODITY)

monthly_panel = monthly_panel.merge(
    hp_long[['year', 'month', 'commodity', 'hp']],
    on=['year', 'month', 'commodity'],
    how='left'
)
monthly_panel = monthly_panel.rename(columns={'hp': 'hp_proxy'})

# 7D. Merge Basis-proxy
basis_long = basis_monthly_df.melt(
    id_vars=['year', 'month', 'month_end'],
    var_name='commodity',
    value_name='basis'
)

monthly_panel = monthly_panel.merge(
    basis_long[['year', 'month', 'commodity', 'basis']],
    on=['year', 'month', 'commodity'],
    how='left'
)
monthly_panel = monthly_panel.rename(columns={'basis': 'basis_proxy'})

print(f"Monthly panel (post-merge): {len(monthly_panel):,} firm-month obs")
print(f"Date range: year {monthly_panel['year'].min()} - {monthly_panel['year'].max()}")
print("\nMissing rates:")
for col in ['bm', 'mom', 'hp_proxy', 'basis_proxy']:
    pct = monthly_panel[col].isna().mean() * 100
    print(f"  {col:<15}: {pct:.1f}% missing")

print(f"\nAvg firms per month: "
      f"{monthly_panel.groupby(['year','month'])['permno'].nunique().mean():.0f}")

Monthly panel (pre-merge): 352,596 firm-month obs
Monthly panel (post-merge): 352,596 firm-month obs
Date range: year 1990 - 2024

Missing rates:
  bm             : 0.0% missing
  mom            : 4.7% missing
  hp_proxy       : 18.9% missing
  basis_proxy    : 13.3% missing

Avg firms per month: 864


---
## 8 · Portfolio Assignments - 7 Overlapping Sort Schemes

Each sort scheme operates on the **same pooled direct-producer universe**.
All sorts use **independent breakpoints** (one firm can appear in multiple sort schemes).

Double sorts (3x3): independent tertile assignment on each dimension.
Single sorts (quintiles): independent quintile assignment.

Breakpoints are computed across the **pooled universe** each month (for monthly-rebalanced
variables) or each June (for BM-only annual sorts).


In [25]:
# Helper: assign tertile / quintile labels

def assign_quantile(series, n_bins, prefix):
    valid = series.dropna()
    n_unique = valid.nunique()

    if n_unique < 2 or len(valid) < n_bins:
        return pd.Series(np.nan, index=series.index)

    if n_unique <= n_bins * 3:
        # Low-cardinality (e.g. hp_proxy with 9 distinct values):
        # bin by distinct value rank so ties don't shift groups up
        unique_vals = sorted(valid.unique())
        n_vals = len(unique_vals)
        val_to_bin = {
            val: min(int(np.floor(i / n_vals * n_bins)) + 1, n_bins)
            for i, val in enumerate(unique_vals)
        }
        return series.map(val_to_bin).astype(float)

    else:
        # High-cardinality continuous variables (bm, mom):
        # equal-frequency binning via firm rank
        ranked = series.rank(method='average', na_option='keep')
        n_obs  = ranked.notna().sum()
        bins   = np.ceil(ranked / n_obs * n_bins).clip(1, n_bins)
        bins[series.isna()] = np.nan
        return bins


def double_sort_labels(grp, var1, var2, n1=3, n2=3, scheme_name=''):
    valid = grp.dropna(subset=[var1, var2, 'me_jun'])
    if len(valid) < n1 * n2:
        return pd.DataFrame()

    valid = valid.copy()
    valid['q1'] = assign_quantile(valid[var1], n1, var1)
    valid['q2'] = assign_quantile(valid[var2], n2, var2)
    valid = valid.dropna(subset=['q1', 'q2'])
    valid['q1'] = valid['q1'].astype(int)
    valid['q2'] = valid['q2'].astype(int)
    valid['port_label'] = (
        scheme_name + '_' +
        valid['q1'].astype(str) + '_' + valid['q2'].astype(str)
    )
    return valid[['permno', 'year', 'month', 'port_label', 'me_jun']]


def single_sort_labels(grp, var1, n1=5, scheme_name=''):
    """
    Single sort. Returns DataFrame with columns
    [permno, year, month, port_label].
    """
    valid = grp.dropna(subset=[var1, 'me_jun'])
    if len(valid) < n1:
        return pd.DataFrame()

    valid = valid.copy()
    valid['q1'] = assign_quantile(valid[var1], n1, var1)
    valid = valid.dropna(subset=['q1'])
    valid['q1'] = valid['q1'].astype(int)
    valid['port_label'] = scheme_name + '_Q' + valid['q1'].astype(str)
    return valid[['permno', 'year', 'month', 'port_label', 'me_jun']]


print("Sort helpers defined.")

Sort helpers defined.


In [ ]:
# 8. Run all 7 sort schemes
#
# Scheme definitions:
#   1. BM x HP-proxy       (3x3, monthly rebalancing)
#   2. MOM x BM            (3x3, monthly rebalancing)
#   3. HP-proxy x MOM      (3x3, monthly rebalancing)
#   4. BM x Basis-proxy    (3x3, BM annual / Basis monthly
#   5. MOM x Basis-proxy   (3x3, MOM monthly / Basis monthly)
#   6. HP-proxy x Basis-proxy (3x3, HP monthly / Basis monthly) - removed due to low coverage
#   7. BM quintiles        (1x5, annual June rebalancing)
#   8. MOM quintiles       (1x5, monthly rebalancing)
#   9. HP quintiles        (1x5, monthly rebalancing)
#   10. Basis quintiles     (1x5, monthly rebalancing)

SORT_SCHEMES = [
    # 6 double sorts: all C(4,2) pairwise combinations
    dict(name='BM_HP',      type='double', v1='bm',          v2='hp_proxy',    n1=5, n2=3),
    dict(name='MOM_BM',     type='double', v1='mom',         v2='bm',          n1=5, n2=5),
    dict(name='HP_MOM',     type='double', v1='hp_proxy',    v2='mom',         n1=3, n2=5),
    dict(name='BM_BASIS',   type='double', v1='bm',          v2='basis_proxy', n1=5, n2=3),
    dict(name='MOM_BASIS',  type='double', v1='mom',         v2='basis_proxy', n1=5, n2=3),
    #dict(name='HP_BASIS',   type='double', v1='hp_proxy',    v2='basis_proxy', n1=3, n2=3),
    # 4 single sorts: one per characteristic
    dict(name='BM',       type='single', v1='bm',                            n1=10),
    dict(name='MOM',      type='single', v1='mom',                           n1=10),
    dict(name='HP',       type='single', v1='hp_proxy',                      n1=5),
    dict(name='BASIS',    type='single', v1='basis_proxy',                   n1=5),
]

all_assignments = []

print("Running sort schemes...")

for scheme in SORT_SCHEMES:
    sname = scheme['name']
    records = []

    for (yr, mo), grp in monthly_panel.groupby(['year', 'month']):
        grp = grp.copy()
        grp['year']  = yr
        grp['month'] = mo

        if scheme['type'] == 'double':
            res = double_sort_labels(
                grp, scheme['v1'], scheme['v2'],
                scheme['n1'], scheme['n2'], sname
            )
        else:
            res = single_sort_labels(
                grp, scheme['v1'], scheme['n1'], sname
            )

        if len(res) > 0:
            records.append(res)

    if records:
        scheme_df = pd.concat(records, ignore_index=True)
        n_ports = scheme_df['port_label'].nunique()
        avg_n   = scheme_df.groupby(['year', 'month', 'port_label']).size().mean()
        print(f"  {sname:<12}: {n_ports} portfolios, avg {avg_n:.0f} firms/cell")
        all_assignments.append(scheme_df)

all_assignments_df = pd.concat(all_assignments, ignore_index=True)

print(f"\nTotal assignments: {len(all_assignments_df):,}")
print(f"Total unique portfolios: {all_assignments_df['port_label'].nunique()}")
print("\nPortfolio labels:")
print(sorted(all_assignments_df['port_label'].unique()))

Running sort schemes...
  BM_HP       : 15 portfolios, avg 56 firms/cell
  MOM_BM      : 25 portfolios, avg 33 firms/cell
  HP_MOM      : 15 portfolios, avg 53 firms/cell
  BM_BASIS    : 15 portfolios, avg 57 firms/cell
  MOM_BASIS   : 15 portfolios, avg 54 firms/cell
  BM          : 10 portfolios, avg 86 firms/cell
  MOM         : 10 portfolios, avg 84 firms/cell
  HP          : 5 portfolios, avg 167 firms/cell
  BASIS       : 5 portfolios, avg 170 firms/cell

Total assignments: 2,776,229
Total unique portfolios: 115

Portfolio labels:
['BASIS_Q1', 'BASIS_Q2', 'BASIS_Q3', 'BASIS_Q4', 'BASIS_Q5', 'BM_BASIS_1_1', 'BM_BASIS_1_2', 'BM_BASIS_1_3', 'BM_BASIS_2_1', 'BM_BASIS_2_2', 'BM_BASIS_2_3', 'BM_BASIS_3_1', 'BM_BASIS_3_2', 'BM_BASIS_3_3', 'BM_BASIS_4_1', 'BM_BASIS_4_2', 'BM_BASIS_4_3', 'BM_BASIS_5_1', 'BM_BASIS_5_2', 'BM_BASIS_5_3', 'BM_HP_1_1', 'BM_HP_1_2', 'BM_HP_1_3', 'BM_HP_2_1', 'BM_HP_2_2', 'BM_HP_2_3', 'BM_HP_3_1', 'BM_HP_3_2', 'BM_HP_3_3', 'BM_HP_4_1', 'BM_HP_4_2', 'BM_HP_4_3', 

---
## 9 · Compute Value-Weighted Daily Returns

Merge monthly sort assignments onto CRSP daily returns.
Use June ME as the VW weight (held constant within the monthly holding period).


In [ ]:
# 9. Daily VW portfolio returns (year-by-year to manage memory)

MIN_STOCKS = 5   # minimum firms for valid VW return

print("Computing daily VW portfolio returns (chunked by year)...")

daily_records = []
years = sorted(dsf['date'].dt.year.unique())

for yr in years:
    # Slice assignments and daily data for this year only
    yr_assign = all_assignments_df[all_assignments_df['year'] == yr][
        ['permno', 'year', 'month', 'port_label', 'me_jun']
    ]
    if len(yr_assign) == 0:
        continue

    yr_dsf = dsf[dsf['date'].dt.year == yr][
        ['permno', 'date', 'year', 'month', 'ret']
    ].copy()

    # Inner merge - only firm-days with a portfolio assignment
    yr_merged = yr_dsf.merge(yr_assign, on=['permno', 'year', 'month'], how='inner')

    # Drop NaN returns before weighting
    yr_merged = yr_merged.dropna(subset=['ret', 'me_jun'])
    yr_merged = yr_merged[yr_merged['me_jun'] > 0]

    # Vectorised VW return: sum(w × r) / sum(w)
    yr_merged['wr'] = yr_merged['me_jun'] * yr_merged['ret']

    grp = yr_merged.groupby(['date', 'port_label'])
    sum_wr = grp['wr'].sum()
    sum_w  = grp['me_jun'].sum()
    counts = grp['ret'].count()

    yr_vw = (sum_wr / sum_w).where(counts >= MIN_STOCKS).rename('ret_vw')
    yr_vw = yr_vw.reset_index()
    daily_records.append(yr_vw)

    del yr_merged, yr_dsf, yr_assign, yr_vw, grp, sum_wr, sum_w, counts
    print(f"  {yr}: done")

daily_port = pd.concat(daily_records, ignore_index=True)

# Pivot to wide format: rows = dates, columns = portfolios
daily_wide = (daily_port
              .pivot(index='date', columns='port_label', values='ret_vw')
              .sort_index())

print(f"\nDaily portfolio returns: {daily_wide.shape[0]} days × "
      f"{daily_wide.shape[1]} portfolios")
print(f"Date range: {daily_wide.index.min().date()} – "
      f"{daily_wide.index.max().date()}")

Computing daily VW portfolio returns (chunked by year)...
  1990: done
  1991: done
  1992: done
  1993: done
  1994: done
  1995: done
  1996: done
  1997: done
  1998: done
  1999: done
  2000: done
  2001: done
  2002: done
  2003: done
  2004: done
  2005: done
  2006: done
  2007: done
  2008: done
  2009: done
  2010: done
  2011: done
  2012: done
  2013: done
  2014: done
  2015: done
  2016: done
  2017: done
  2018: done
  2019: done
  2020: done
  2021: done
  2022: done
  2023: done

Daily portfolio returns: 8439 days × 115 portfolios
Date range: 1990-07-02 – 2023-12-29


In [28]:
# 9B. Compute VW return each day per portfolio
MIN_STOCKS = MIN_STOCKS_DOUBLE  # use the double-sort minimum

def vw_return(df):
    """Value-weighted return with minimum firm count guard."""
    if len(df) < MIN_STOCKS:
        return np.nan
    w = df['me_jun']
    if w.sum() == 0:
        return np.nan
    return (w * df['ret']).sum() / w.sum()

print("Computing daily VW portfolio returns...")

daily_port = pd.concat(daily_records, ignore_index=True)
print(f"\nDaily long format: {len(daily_port):,} rows")
print(f"Portfolios: {daily_port['port_label'].nunique()}")

Computing daily VW portfolio returns...

Daily long format: 895,878 rows
Portfolios: 115


---
## 10 · Aggregate Daily to Weekly Returns

Use **Wednesday-to-Wednesday** windows (`W-WED`).
A weekly return is valid only if at least **3 trading days** are available.


In [ ]:
# 10. Aggregate daily to weekly (Wednesday close)
print("Aggregating daily to weekly (Wednesday close)...")

# Pivot to wide only after ensuring daily_port is the correct size
daily_wide = (daily_port
              .pivot_table(index='date', columns='port_label',
                           values='ret_vw', aggfunc='first')
              .sort_index())

# Vectorised log-return compounding - avoids resample().apply() memory spike
log_daily  = np.log1p(daily_wide)
valid_days = daily_wide.notna()
week_id    = daily_wide.index.to_period('W-WED')

log_weekly  = log_daily.groupby(week_id).sum(min_count=1)
n_valid     = valid_days.groupby(week_id).sum()

weekly_wide = np.expm1(log_weekly)
weekly_wide[n_valid < 3] = np.nan
weekly_wide.index = weekly_wide.index.to_timestamp(how='end').normalize()
weekly_wide.index.name = 'date'

print(f"Weekly portfolio returns: {weekly_wide.shape[0]} weeks × "
      f"{weekly_wide.shape[1]} portfolios")
print(f"Date range: {weekly_wide.index.min().date()} – "
      f"{weekly_wide.index.max().date()}")

cov = weekly_wide.notna().mean().sort_values()
print(f"\nPortfolio coverage (fraction non-missing):")
print(cov.round(2).to_string())

Aggregating daily to weekly (Wednesday close)...
Weekly portfolio returns: 1749 weeks × 115 portfolios
Date range: 1990-07-04 – 2024-01-03

Portfolio coverage (fraction non-missing):
port_label
HP_MOM_2_1      0.84
HP_MOM_2_5      0.84
HP_MOM_2_3      0.85
HP_MOM_1_1      0.85
BM_HP_1_2       0.85
HP_MOM_2_2      0.85
HP_MOM_2_4      0.85
BM_HP_2_2       0.85
BM_HP_4_2       0.85
BM_HP_5_2       0.85
BM_HP_3_2       0.85
HP_MOM_3_1      0.85
BM_HP_1_3       0.85
BM_HP_1_1       0.85
BM_HP_2_1       0.85
BM_HP_2_3       0.85
HP_MOM_3_2      0.85
HP_MOM_3_3      0.85
HP_MOM_1_3      0.85
HP_MOM_1_2      0.85
BM_HP_5_1       0.85
BM_HP_5_3       0.85
HP_MOM_1_4      0.85
HP_MOM_1_5      0.85
BM_HP_4_3       0.85
BM_HP_4_1       0.85
BM_HP_3_3       0.85
BM_HP_3_1       0.85
HP_MOM_3_5      0.85
HP_MOM_3_4      0.85
HP_Q2           0.85
HP_Q1           0.85
HP_Q3           0.85
HP_Q4           0.85
HP_Q5           0.85
MOM_BASIS_1_2   0.87
MOM_BASIS_3_2   0.89
MOM_BASIS_2_2   0.89
MOM_BASI

---
## 11 · Subtract Risk-Free Rate --> Excess Returns


In [ ]:
print("Downloading daily Fama-French risk-free rate...")

rf_url   = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/"
            "ftp/F-F_Research_Data_Factors_daily_CSV.zip")
response = urllib.request.urlopen(rf_url)
zf       = zipfile.ZipFile(io.BytesIO(response.read()))
fname    = zf.namelist()[0]

def compound_returns(series):
    """Compound a series of simple returns (e.g. daily RF --> weekly RF)."""
    valid = series.dropna()
    if len(valid) == 0:
        return np.nan
    return (1 + valid).prod() - 1

ff_daily = pd.read_csv(zf.open(fname), skiprows=4, index_col=0)
valid_idx = ff_daily.index.astype(str).str.match(r'^\d{8}$')
ff_daily  = ff_daily[valid_idx].copy()
ff_daily.index = pd.to_datetime(ff_daily.index.astype(str), format='%Y%m%d')
ff_daily['RF']  = ff_daily['RF'].astype(float) / 100

rf_weekly = (
    ff_daily['RF']
    .resample('W-WED')
    .apply(compound_returns)
    .rename('rf')
)

print(f"RF weekly: {len(rf_weekly)} weeks, "
      f"{rf_weekly.index.min().date()} - {rf_weekly.index.max().date()}")
print(f"Mean annual RF: {rf_weekly.mean() * 52 * 100:.2f}%")

# Subtract RF and restrict to study period (1995 driven by CFTC COT availability)
# Note: weekly_wide.index was normalised in cell 10 (.normalize()), so it aligns
# cleanly with rf_weekly's resample-based W-WED index.
weekly_excess = (weekly_wide
                 .subtract(rf_weekly, axis=0)
                 .loc['1995-01-01':'2023-12-31']
                 .dropna(how='all'))

print(f"\nBlock 2 excess returns:")
print(f"  Shape : {weekly_excess.shape[0]:,} weeks x {weekly_excess.shape[1]} portfolios")
print(f"  Period: {weekly_excess.index.min().date()} - {weekly_excess.index.max().date()}")
print(f"\nAnnualised mean excess return by portfolio (%):")
ann = (weekly_excess.mean() * 52 * 100).sort_values(ascending=False)
print(ann.round(2).to_string())

RF weekly: 5210 weeks, 1926-07-07 - 2026-05-06
Mean annual RF: 3.13%

Block 2 excess returns:
  Shape : 1,513 weeks x 115 portfolios
  Period: 1995-01-04 - 2023-12-27

Annualised mean excess return by portfolio (%):
port_label
BM_HP_5_3       22.29
BM_Q10          20.50
MOM_Q1          20.26
HP_MOM_2_1      17.11
MOM_BM_1_1      16.74
MOM_BASIS_2_2   16.49
BM_BASIS_5_1    16.20
MOM_BM_1_5      16.08
MOM_BASIS_1_3   15.94
MOM_BM_2_1      15.58
MOM_BM_5_5      15.48
HP_MOM_2_2      15.44
HP_Q4           15.11
MOM_BM_1_2      15.04
BM_HP_5_2       14.55
BM_HP_3_3       14.31
HP_MOM_3_5      14.30
BM_HP_4_3       13.91
MOM_Q10         13.88
BM_BASIS_5_2    13.86
MOM_BASIS_1_2   13.56
BASIS_Q1        13.45
HP_MOM_3_3      13.32
HP_MOM_3_1      13.27
BM_BASIS_5_3    13.16
MOM_BM_3_5      13.15
MOM_BM_4_5      13.14
BM_BASIS_4_1    13.09
BM_HP_2_3       12.74
BM_Q9           12.74
MOM_BASIS_1_1   12.72
MOM_BASIS_4_1   12.45
MOM_BM_5_4      12.41
HP_MOM_3_4      12.38
MOM_BM_2_2      12.34
MOM

---
## 12 · Diagnostics


In [31]:
# Summary statistics by sort scheme
print("Block 2 summary statistics (weekly excess returns, annualised):")
print("="*70)

for scheme in SORT_SCHEMES:
    sname  = scheme['name']
    cols   = [c for c in weekly_excess.columns if c.startswith(sname + '_')]
    if not cols:
        continue
    sub = weekly_excess[cols]
    means = (sub.mean() * 52 * 100).values
    print(f"  {sname:<12}  N={len(cols):2d}  "
          f"mean={np.nanmean(means):.1f}%  "
          f"min={np.nanmin(means):.1f}%  "
          f"max={np.nanmax(means):.1f}%")

print(f"\nTotal Block 2 portfolios: {len(weekly_excess.columns)}")
print(f"(Target: 115 = 5 double sorts [15+25+15+15+15] + 4 single sorts [10+10+5+5])")

# Stock count diagnostics
counts = (
    all_assignments_df
    .groupby(['year', 'month', 'port_label'])
    .size()
    .reset_index(name='n_stocks')
)

print("\nStock count per portfolio (time average across all months):")
avg_counts = counts.groupby('port_label')['n_stocks'].mean().sort_values()
print(f"  Min: {avg_counts.min():.0f}  Median: {avg_counts.median():.0f}  "
      f"Max: {avg_counts.max():.0f}")

thin_ports = avg_counts[avg_counts < MIN_STOCKS_DOUBLE]
if len(thin_ports) > 0:
    print(f"\nWARNING: {len(thin_ports)} portfolios average < {MIN_STOCKS_DOUBLE} stocks:")
    print(thin_ports.to_string())
else:
    print(f"\nAll portfolios average ≥ {MIN_STOCKS_DOUBLE} stocks per period.")

Block 2 summary statistics (weekly excess returns, annualised):
  BM_HP         N=15  mean=10.4%  min=2.8%  max=22.3%
  MOM_BM        N=25  mean=11.4%  min=5.1%  max=16.7%
  HP_MOM        N=15  mean=10.3%  min=4.5%  max=17.1%
  BM_BASIS      N=15  mean=11.1%  min=8.9%  max=16.2%
  MOM_BASIS     N=15  mean=11.4%  min=6.5%  max=16.5%
  BM            N=40  mean=10.8%  min=2.8%  max=22.3%
  MOM           N=50  mean=11.4%  min=5.1%  max=20.3%
  HP            N=20  mean=10.0%  min=2.2%  max=17.1%
  BASIS         N= 5  mean=8.7%  min=4.1%  max=13.4%

Total Block 2 portfolios: 115
(Target: 115 = 5 double sorts [15+25+15+15+15] + 4 single sorts [10+10+5+5])

Stock count per portfolio (time average across all months):
  Min: 25  Median: 55  Max: 223

All portfolios average ≥ 5 stocks per period.


---
## 13 · Save Outputs


In [ ]:
# Block 2 weekly excess returns
# Column naming convention:
#   Double sorts: {SCHEME}_{q1}_{q2}   e.g. BM_HP_1_3
#   Single sorts: {SCHEME}_Q{q}        e.g. BM_Q_5

weekly_excess.index.name = 'date'
weekly_excess = weekly_excess.loc['1995-07-05':].copy()
weekly_excess.fillna(0).to_csv('Output/commodity_equity_weekly_by_characteristics.csv', index=True)

print(f"Saved Output/commodity_equity_weekly_by_characteristics.csv  "
      f"({weekly_excess.shape[0]} weeks x {weekly_excess.shape[1]} portfolios)")
print(weekly_excess.head())

# Block 2 stock counts
counts.to_csv('Output/portfolio_counts.csv', index=False)
print(f"Saved Output/portfolio_counts.csv  ({len(counts):,} rows)")

Saved commodity_equity_weekly_by_characteristics.csv  (1487 weeks x 115 portfolios)
port_label  BASIS_Q1  BASIS_Q2  BASIS_Q3  BASIS_Q4  BASIS_Q5  BM_BASIS_1_1  \
date                                                                         
1995-07-05      0.02     -0.01      0.01     -0.00      0.01          0.00   
1995-07-12      0.02      0.02      0.08     -0.00      0.03          0.02   
1995-07-19     -0.01     -0.02     -0.05     -0.00     -0.03         -0.01   
1995-07-26      0.01      0.01      0.00      0.01     -0.00          0.02   
1995-08-02     -0.01     -0.01      0.01      0.01      0.00         -0.01   

port_label  BM_BASIS_1_2  BM_BASIS_1_3  BM_BASIS_2_1  BM_BASIS_2_2  \
date                                                                 
1995-07-05         -0.00         -0.01          0.00          0.01   
1995-07-12          0.01         -0.01          0.01          0.01   
1995-07-19         -0.01          0.01         -0.01         -0.03   
1995-07-26         

In [33]:
df = pd.read_csv('Output/commodity_equity_weekly_by_characteristics.csv', parse_dates=['date'])
data = df.drop(columns='date')
nan_mask = data.isna()

def get_scheme(col):
    if col.startswith('BASIS_Q'):   return 'BASIS_Q'
    if col.startswith('BM_BASIS'):  return 'BM_BASIS'
    if col.startswith('BM_HP'):     return 'BM_HP'
    if col.startswith('BM_Q'):      return 'BM_Q'
    if col.startswith('HP_MOM'):    return 'HP_MOM'
    if col.startswith('HP_Q'):      return 'HP_Q'
    if col.startswith('MOM_BASIS'): return 'MOM_BASIS'
    if col.startswith('MOM_BM'):    return 'MOM_BM'
    if col.startswith('MOM_Q'):     return 'MOM_Q'
    return 'OTHER'

from itertools import groupby
schemes = {}
for c in data.columns:
    schemes.setdefault(get_scheme(c), []).append(c)

print(f"{'Scheme':<12} {'Cols':>5} {'NaN%':>7} {'Fully clean':>12}")
print("-" * 42)
for scheme, cols in sorted(schemes.items()):
    total = sum(nan_mask[c].sum() for c in cols)
    pct   = total / (len(cols) * len(df)) * 100
    clean = sum(nan_mask[c].sum() == 0 for c in cols)
    print(f"{scheme:<12} {len(cols):>5} {pct:>6.1f}% {clean:>10}/{len(cols)}")
print(f"\nTotal portfolios: {sum(len(v) for v in schemes.values())}")

Scheme        Cols    NaN%  Fully clean
------------------------------------------
BASIS_Q          5    0.0%          5/5
BM_BASIS        15    0.0%         15/15
BM_HP           15    0.0%         15/15
BM_Q            10    0.0%         10/10
HP_MOM          15    0.0%         15/15
HP_Q             5    0.0%          5/5
MOM_BASIS       15    0.0%         15/15
MOM_BM          25    0.0%         25/25
MOM_Q           10    0.0%         10/10

Total portfolios: 115


In [ ]:
# Section 14 · Commodity Factors (long-short spreads for Pass 3)
#
# Constructs four zero-cost factor portfolios from the single-sort quintile/
# decile columns already computed above. Each factor = top quintile/decile
# minus bottom quintile/decile of the relevant characteristic.
#
# Factor definitions:
#   BM    : BM_Q10 - BM_Q1   (high value minus low value)
#   MOM   : MOM_Q10 - MOM_Q1 (high momentum minus low momentum)
#   HP    : HP_Q5  - HP_Q1   (high hedging pressure minus low)
#   BASIS : BASIS_Q5 - BASIS_Q1 (high basis = backwardation minus low = contango)

commodity_factors = pd.DataFrame({
    'BM':    weekly_excess['BM_Q10']    - weekly_excess['BM_Q1'],    # value premium
    'MOM':   weekly_excess['MOM_Q10']   - weekly_excess['MOM_Q1'],   # reversal finding
    'HP':    weekly_excess['HP_Q5']     - weekly_excess['HP_Q1'],    # hedging pressure premium
    'BASIS': weekly_excess['BASIS_Q1']  - weekly_excess['BASIS_Q5'], # backwardation premium
}, index=weekly_excess.index)

commodity_factors.index.name = 'date'
commodity_factors.to_csv('Output/commodity_factors_weekly.csv')

print('Saved commodity_factors_weekly.csv')
print(f'  T={len(commodity_factors)}, factors={commodity_factors.columns.tolist()}')
print(f'\nFactor summary (annualised %pa):')
print(f'  {"Factor":<8}  {"Mean":>8}  {"Std":>8}  {"Sharpe":>8}')
print('  '+'-'*36)
for col in commodity_factors.columns:
    m  = commodity_factors[col].mean() * 52 * 100
    s  = commodity_factors[col].std()  * np.sqrt(52) * 100
    sr = m / s if s > 0 else np.nan
    print(f'  {col:<8}  {m:>8.2f}%  {s:>8.2f}%  {sr:>8.3f}')

Saved commodity_factors_weekly.csv
  T=1487, factors=['BM', 'MOM', 'HP', 'BASIS']

Factor summary (annualised %pa):
  Factor        Mean       Std    Sharpe
  ------------------------------------
  BM           10.91%     30.19%     0.361
  MOM          -5.38%     45.89%    -0.117
  HP            7.40%     18.98%     0.390
  BASIS         5.46%     18.79%     0.290
